In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub

/kaggle/input/datasets/ladkevinkumar/world-population/world_population.csv


In [2]:
# Install compatible versions without changing Kaggle dependencies
%pip install -q --no-deps "plotly==5.24.1" "kaleido==0.2.1"

Note: you may need to restart the kernel to use updated packages.


In [4]:
import plotly
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Create folder for PNG charts
OUTPUT_DIR = Path("/kaggle/working/quiz_charts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


CONTINENT_COLORS = {
    "Africa": ORANGE,
    "Asia": BLUE,
    "Europe": PURPLE,
    "North America": "#56B4E9",
    "South America": "#7B61A8",
    "Oceania": GREY
}

print("Plotly version:", plotly.__version__)
print("PNG folder created:", OUTPUT_DIR)

Plotly version: 5.24.1
PNG folder created: /kaggle/working/quiz_charts


In [5]:
# Use your exact Kaggle dataset path
csv_path = (
    "/kaggle/input/datasets/ladkevinkumar/world-population/world_population.csv"
)

df = pd.read_csv(csv_path)

# Rename columns
df = df.rename(columns={
    "Country/Territory": "Country",
    "Area (km²)": "Area",
    "Density (per km²)": "Density",
    "World Population Percentage": "World share (%)"
})

# Convert population columns into long format
year_cols = sorted(
    [c for c in df.columns if c.endswith("Population")],
    key=lambda c: int(c.split()[0])
)

df_long = df.melt(
    id_vars=["CCA3", "Country", "Continent"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Population"
)

df_long["Year"] = df_long["Year"].str[:4].astype(int)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Dataset shape: (234, 16)


,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area,Density,World share (%)
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,0.00


In [6]:
def finish_chart(fig, filename):

    fig.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=14, color=DARK),
        title=dict(x=0.02, font=dict(size=21)),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1200,
        height=675,
        margin=dict(l=80, r=100, t=110, b=80)
    )

    fig.show()

    # Save PNG image
    file_path = OUTPUT_DIR / filename

    fig.write_image(
        str(file_path),
        format="png",
        width=1600,
        height=900
    )

    print("Saved:", file_path)

**Q1: Which continent is growing fastest?**

**Answer:** A line chart shows how each continent's population changed from 1970 to 2022. Indexing the population to 100 in 1970 allows us to compare relative growth fairly, even though continents have very different population sizes.

In [7]:
# Aggregate population by continent and year
q1 = df_long.groupby(
    ["Continent", "Year"],
    as_index=False
)["Population"].sum()

# Get baseline population for 1970
baseline = (
    q1[q1["Year"] == 1970]
    [["Continent", "Population"]]
    .rename(columns={"Population": "Baseline"})
)

q1 = q1.merge(baseline, on="Continent")

# Calculate growth index (1970 = 100)
q1["Growth Index"] = (
    q1["Population"] / q1["Baseline"] * 100
)

# Identify fastest growing continent
latest = q1[q1["Year"] == 2022].copy()

fastest = latest.loc[
    latest["Growth Index"].idxmax(), "Continent"
]

# Draw line chart
fig = px.line(
    q1,
    x="Year",
    y="Growth Index",
    color="Continent",
    markers=True,
    color_discrete_map={
        c: ORANGE if c == fastest else GREY
        for c in q1["Continent"].unique()
    },
    title=(
        f"{fastest} experienced the fastest relative "
        "population growth (1970–2022)"
    ),
    template="plotly_white"
)

# Highlight fastest-growing continent
for trace in fig.data:
    if trace.name == fastest:
        trace.update(
            line=dict(width=4),
            marker=dict(size=8)
        )
    else:
        trace.update(
            line=dict(width=1.5),
            marker=dict(size=5),
            opacity=0.5
        )

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Population index (1970 = 100)",
    legend_title="Continent"
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, rangemode="tozero")

# Save chart as PNG
finish_chart(fig, "Q1_continent_growth.png")

# Print interpretation values
print("Fastest-growing continent:", fastest)

print(
    latest[["Continent", "Growth Index"]]
    .sort_values("Growth Index", ascending=False)
    .round(2)
)

Saved: /kaggle/working/quiz_charts/Q1_continent_growth.png
Fastest-growing continent: Africa
        Continent  Growth Index
7          Africa        390.41
39        Oceania        231.20
47  South America        226.39
15           Asia        220.12
31  North America        190.31
23         Europe        113.30


**Q2: Which countries added the most people?**

**Answer:** Horizontal bars make it easy to rank the 15 countries with the largest population increases. African countries are highlighted in orange so their contribution can be identified immediately.

In [8]:

q2 = df.copy()

# Calculate population added between 1970 and 2022
q2["Population Added"] = (
    q2["2022 Population"] -
    q2["1970 Population"]
)

# Convert to millions
q2["Added Millions"] = (
    q2["Population Added"] / 1e6
)

# Select the 15 largest increases
top15 = q2.nlargest(
    15, "Population Added"
).sort_values("Population Added")

# Identify African countries
top15["Group"] = np.where(
    top15["Continent"] == "Africa",
    "Africa",
    "Other continents"
)

africa_count = (
    top15["Continent"] == "Africa"
).sum()

# Create horizontal bar chart
fig = px.bar(
    top15,
    x="Added Millions",
    y="Country",
    orientation="h",
    color="Group",
    text="Added Millions",
    color_discrete_map={
        "Africa": ORANGE,
        "Other continents": GREY
    },
    title=(
        "India and China added the most people "
        f"since 1970; {africa_count} African countries "
        "rank in the top 15"
    ),
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}M",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    xaxis_title="Population added (millions)",
    yaxis_title=None,
    legend_title=None
)

fig.update_xaxes(
    showgrid=False,
    range=[0, top15["Added Millions"].max() * 1.2]
)
fig.update_yaxes(showgrid=False)

# Save chart as PNG
finish_chart(fig, "Q2_top15_population_increase.png")

print("African countries in top 15:", africa_count)

print(
    top15[[
        "Country", "Continent", "Added Millions"
    ]]
    .sort_values("Added Millions", ascending=False)
    .round(2)
)

Saved: /kaggle/working/quiz_charts/Q2_top15_population_increase.png
African countries in top 15: 4
           Country      Continent  Added Millions
92           India           Asia          859.67
41           China           Asia          603.35
156       Pakistan           Asia          176.53
149        Nigeria         Africa          162.97
93       Indonesia           Asia          160.27
221  United States  North America          137.96
27          Brazil  South America          118.94
16      Bangladesh           Asia          103.64
63        Ethiopia         Africa           95.07
55        DR Congo         Africa           78.86
163    Philippines           Asia           78.12
131         Mexico  North America           77.21
57           Egypt         Africa           76.21
94            Iran           Asia           60.10
228        Vietnam           Asia           56.26


**Q3: Do bigger countries have more people?**

**Answer:** A scatter plot reveals the relationship between land area and population. Logarithmic axes improve visibility across very small and very large countries, while continent colors and selected labels make unusual patterns easier to identify.

In [9]:

q3 = df[
    (df["Area"] > 0) &
    (df["2022 Population"] > 0)
].copy()

# Population in millions
q3["Population Millions"] = (
    q3["2022 Population"] / 1e6
)

# Create scatter plot
fig = px.scatter(
    q3,
    x="Area",
    y="Population Millions",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    opacity=0.7,
    color_discrete_map=CONTINENT_COLORS,
    title=(
        "Large land area does not always mean "
        "a large population in 2022"
    ),
    template="plotly_white"
)

fig.update_traces(
    marker=dict(size=9)
)

# Highlight important outlier countries
selected_countries = [
    "India",
    "China",
    "Russia",
    "Canada",
    "Australia",
    "Bangladesh"
]

for _, row in q3[
    q3["Country"].isin(selected_countries)
].iterrows():

    fig.add_annotation(
        x=row["Area"],
        y=row["Population Millions"],
        text=row["Country"],
        showarrow=True,
        ax=25,
        ay=-25,
        arrowhead=0,
        font=dict(size=11, color=DARK),
        arrowcolor=GREY
    )

fig.update_layout(
    xaxis_title="Land area (km², log scale)",
    yaxis_title="Population (millions, log scale)",
    legend_title="Continent"
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

# Save PNG
finish_chart(fig, "Q3_area_vs_population.png")

# Calculate relationship using logarithmic values
correlation = np.corrcoef(
    np.log10(q3["Area"]),
    np.log10(q3["2022 Population"])
)[0, 1]

print("Log-scale Pearson correlation:", round(correlation, 3))

Saved: /kaggle/working/quiz_charts/Q3_area_vs_population.png
Log-scale Pearson correlation: 0.872


**Q4: How crowded are the continents?**

**Answer:** A box plot shows the median, interquartile range, and outliers of population density. This helps compare how density varies both within and between continents, rather than looking only at averages.

In [10]:
q4 = df[df["Density"] > 0].copy()

# Calculate median population density
median_density = (
    q4.groupby("Continent")["Density"]
    .median()
    .sort_values(ascending=False)
)

continent_order = median_density.index.tolist()
highest = continent_order[0]

# Create box plot
fig = px.box(
    q4,
    x="Continent",
    y="Density",
    color="Continent",
    category_orders={
        "Continent": continent_order
    },
    color_discrete_map={
        c: ORANGE if c == highest else GREY
        for c in continent_order
    },
    points="outliers",
    log_y=True,
    hover_name="Country",
    title=(
        f"{highest} has the highest median "
        "country-level population density"
    ),
    template="plotly_white"
)

fig.update_traces(
    line=dict(width=1.5),
    marker=dict(size=5, opacity=0.6)
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title="People per km² (log scale)",
    showlegend=False
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

# Save PNG
finish_chart(fig, "Q4_population_density.png")

print("Median density by continent:")
print(median_density.round(2))

Saved: /kaggle/working/quiz_charts/Q4_population_density.png
Median density by continent:
Continent
North America    221.42
Asia             122.97
Europe           106.87
Oceania           78.25
Africa            59.71
South America     17.79
Name: Density, dtype: float64


**Q5: Where did populations grow and shrink?**

**Answer:** A choropleth map displays population changes geographically, helping identify where countries experienced growth or decline. A diverging color scale makes positive and negative changes visually distinct.

In [11]:
q5 = df.copy()

# Calculate population percentage change
q5["Change (%)"] = (
    (
        q5["2022 Population"] -
        q5["2000 Population"]
    ) / q5["2000 Population"]
) * 100

# Count growth and decline
growing = (q5["Change (%)"] > 0).sum()
shrinking = (q5["Change (%)"] < 0).sum()

# Cap extreme values for clearer map colors
q5["Map Change"] = (
    q5["Change (%)"].clip(-50, 150)
)

# Diverging scale with zero at 25%
map_colors = [
    [0.00, "#2166AC"],
    [0.15, "#92B4D5"],
    [0.25, "#E0E0E0"],
    [0.55, "#F3C567"],
    [1.00, "#E69F00"]
]

# Create choropleth map
fig = px.choropleth(
    q5,
    locations="CCA3",
    locationmode="ISO-3",
    color="Map Change",
    hover_name="Country",
    custom_data=["Change (%)"],
    color_continuous_scale=map_colors,
    range_color=(-50, 150),
    title=(
        f"{growing} countries and territories grew, "
        f"while {shrinking} shrank (2000–2022)"
    ),
    template="plotly_white"
)

fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b>"
        "<br>Population change: %{customdata[0]:+.1f}%"
        "<extra></extra>"
    )
)

fig.update_geos(
    showframe=False,
    showcoastlines=True,
    coastlinecolor="#BBBBBB",
    showcountries=True,
    countrycolor="white",
    projection_type="natural earth"
)

fig.update_layout(
    coloraxis_colorbar=dict(
        title="Change (%)",
        tickvals=[-50, 0, 50, 100, 150],
        ticktext=[
            "≤−50", "0", "+50", "+100", "≥+150"
        ]
    )
)

# Save PNG
finish_chart(fig, "Q5_world_population_map.png")

print("Countries/territories with growth:", growing)
print("Countries/territories with decline:", shrinking)

print("\nCountries with largest percentage decline:")
print(
    q5.nsmallest(10, "Change (%)")[
        ["Country", "Change (%)"]
    ].round(2)
)

Saved: /kaggle/working/quiz_charts/Q5_world_population_map.png
Countries/territories with growth: 200
Countries/territories with decline: 34

Countries with largest percentage decline:
                      Country  Change (%)
153  Northern Mariana Islands      -38.32
3              American Samoa      -23.97
117                 Lithuania      -23.60
126          Marshall Islands      -23.34
133                   Moldova      -23.02
111                    Latvia      -22.65
25     Bosnia and Herzegovina      -22.63
226              Vatican City      -21.66
229         Wallis and Futuna      -21.40
218                   Ukraine      -18.78


**Q6: Where do the world's 8 billion people live?**

**Answer:** A treemap shows hierarchical relationships. Continent rectangles contain country rectangles, and their area represents population. This makes it possible to compare global population distribution at both levels in one figure.

In [12]:
q6 = df.copy()

# Total world population
world_total = q6["2022 Population"].sum()

# Calculate share
q6["World Share (%)"] = (
    q6["2022 Population"] / world_total * 100
)

# Calculate continent shares
continent_shares = (
    q6.groupby("Continent")["2022 Population"]
    .sum()
    .div(world_total)
    .mul(100)
    .sort_values(ascending=False)
)

largest_continent = continent_shares.index[0]
largest_share = continent_shares.iloc[0]

# Create treemap
fig = px.treemap(
    q6,
    path=["Continent", "Country"],
    values="2022 Population",
    color="Continent",
    color_discrete_map=CONTINENT_COLORS,
    title=(
        f"{largest_continent} contains {largest_share:.1f}% "
        "of the world's population in 2022"
    ),
    template="plotly_white"
)

fig.update_traces(
    textinfo="label+percent root",
    textfont=dict(size=14),
    marker=dict(
        line=dict(color="white", width=1)
    ),
    root_color="#EEEEEE",
    hovertemplate=(
        "<b>%{label}</b>"
        "<br>Population: %{value:,.0f}"
        "<br>World share: %{percentRoot:.2%}"
        "<extra></extra>"
    )
)

fig.update_layout(showlegend=False)

# Save PNG
finish_chart(fig, "Q6_world_population_treemap.png")

print("World population:", f"{world_total:,.0f}")

print("\nPopulation share by continent (%):")
print(continent_shares.round(2))

Saved: /kaggle/working/quiz_charts/Q6_world_population_treemap.png
World population: 7,973,413,042

Population share by continent (%):
Continent
Asia             59.21
Africa           17.89
Europe            9.32
North America     7.53
South America     5.48
Oceania           0.56
Name: 2022 Population, dtype: float64


**Q7: Whose share of the world is rising?**

**Answer:** A 100% stacked bar chart compares how each region's share of world population changes over time. Every bar represents the entire world population for that year, making shifts in proportions clear.

In [13]:
q7 = df_long.copy()

# Group other three continents
q7["Region"] = q7["Continent"].where(
    q7["Continent"].isin([
        "Africa", "Asia", "Europe"
    ]),
    "Rest of the world"
)

# Aggregate population
q7 = q7.groupby(
    ["Year", "Region"],
    as_index=False
)["Population"].sum()

# Total world population for each year
q7["World Total"] = q7.groupby(
    "Year"
)["Population"].transform("sum")

# Calculate percentage shares
q7["Share (%)"] = (
    q7["Population"] / q7["World Total"] * 100
)

region_order = [
    "Asia",
    "Africa",
    "Europe",
    "Rest of the world"
]

region_colors = {
    "Asia": BLUE,
    "Africa": ORANGE,
    "Europe": PURPLE,
    "Rest of the world": GREY
}

# Create stacked bar chart
fig = px.bar(
    q7,
    x="Year",
    y="Share (%)",
    color="Region",
    barmode="stack",
    category_orders={"Region": region_order},
    color_discrete_map=region_colors,
    title=(
        "Africa's world population share is rising "
        "while Europe's is falling (1970–2022)"
    ),
    template="plotly_white"
)

fig.update_traces(
    hovertemplate=(
        "%{fullData.name}"
        "<br>Year: %{x}"
        "<br>Population share: %{y:.1f}%"
        "<extra></extra>"
    )
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Share of world population (%)",
    legend_title=None,
    legend=dict(
        orientation="h",
        y=-0.22,
        x=0.5,
        xanchor="center"
    ),
    bargap=0.2
)

fig.update_xaxes(
    type="category",
    showgrid=False
)

fig.update_yaxes(
    range=[0, 100],
    ticksuffix="%",
    showgrid=False
)

# Save PNG
finish_chart(fig, "Q7_world_population_share.png")

# Compare first and last observation years
comparison = q7[
    q7["Year"].isin([1970, 2022])
].pivot(
    index="Region",
    columns="Year",
    values="Share (%)"
)

print("Population shares (%) in 1970 and 2022:")
display(comparison.round(2))

Saved: /kaggle/working/quiz_charts/Q7_world_population_share.png
Population shares (%) in 1970 and 2022:


Year,1970,2022
Region,,
Africa,9.89,17.89
Asia,58.06,59.21
Europe,17.76,9.32
Rest of the world,14.29,13.57
